# NB10 — compbio_utils Package: IO and Data Utilities

**Module 16 — Research Software Engineering**

---

## Motivation

This notebook builds `compbio_utils/io.py` — the file I/O module. Bioinformatics pipelines spend a surprising fraction of their time on file parsing: reading FASTA, FASTQ, BED, and VCF files. A well-tested parser catches malformed files early, with useful error messages, rather than propagating bad data silently through a pipeline.

## Biological Background

- **FASTA**: the simplest sequence format — a header line starting with `>`, followed by sequence lines. Used for reference genomes, protein databases, assembled contigs.
- **FASTQ**: extends FASTA with per-base quality scores (Phred encoding). Used for raw reads from Illumina, PacBio, Nanopore sequencers.
- **BED**: Browser Extensible Data format. Tab-separated: chromosome, start (0-based), end (exclusive), optional name/score/strand. Used for genomic interval annotations (genes, peaks, repeats).

---

In [ ]:
from __future__ import annotations
from pathlib import Path
import io
import pandas as pd


VALID_ALPHABETS: dict[str, frozenset[str]] = {
    'DNA': frozenset('ATCGatcgNn'),
    'RNA': frozenset('AUCGaucgNn'),
    'protein': frozenset('ACDEFGHIKLMNPQRSTVWYacdefghiklmnpqrstvwy*'),
}


def validate_sequence(seq: str, alphabet: str = 'DNA') -> bool:
    """Validate that a sequence contains only valid characters.

    Parameters
    ----------
    seq : str
        Sequence to validate.
    alphabet : {'DNA', 'RNA', 'protein'}, optional
        Expected alphabet. Default is 'DNA'.

    Returns
    -------
    bool
        True if all characters are in the alphabet, False otherwise.

    Raises
    ------
    ValueError
        If ``alphabet`` is not one of the supported options.

    Examples
    --------
    >>> validate_sequence('ATCG')
    True
    >>> validate_sequence('ATCGN')
    True
    >>> validate_sequence('ATCGZ')
    False
    """
    if alphabet not in VALID_ALPHABETS:
        raise ValueError(f'alphabet must be one of {list(VALID_ALPHABETS)}, got {alphabet!r}')
    valid = VALID_ALPHABETS[alphabet]
    return all(c in valid for c in seq)


def read_fasta(filepath: str | Path) -> dict[str, str]:
    """Parse a FASTA file and return sequences as a dictionary.

    Parameters
    ----------
    filepath : str or Path
        Path to the FASTA file.

    Returns
    -------
    dict[str, str]
        Mapping of sequence header (without '>') to sequence string.
        Multi-line sequences are concatenated. Order matches file order.

    Raises
    ------
    FileNotFoundError
        If ``filepath`` does not exist.
    ValueError
        If the file is empty or does not start with '>'.

    Examples
    --------
    >>> # Write a temp FASTA and read it back
    >>> import tempfile, pathlib
    >>> with tempfile.NamedTemporaryFile(mode='w', suffix='.fa', delete=False) as f:
    ...     _ = f.write('>seq1\nATCG\n>seq2\nGGGG\n')
    ...     tmp = f.name
    >>> seqs = read_fasta(tmp)
    >>> seqs['seq1']
    'ATCG'
    """
    filepath = Path(filepath)
    if not filepath.exists():
        raise FileNotFoundError(f'FASTA file not found: {filepath}')

    sequences: dict[str, str] = {}
    current_header: str | None = None
    current_seq_parts: list[str] = []

    with filepath.open() as fh:
        for line_num, line in enumerate(fh, start=1):
            line = line.rstrip('\n')
            if not line:
                continue  # skip blank lines
            if line.startswith('>'):
                # Save previous sequence
                if current_header is not None:
                    sequences[current_header] = ''.join(current_seq_parts)
                current_header = line[1:].split()[0]  # use first word of header
                current_seq_parts = []
            else:
                if current_header is None:
                    raise ValueError(f'Line {line_num}: sequence before first header')
                current_seq_parts.append(line.strip())

    # Save last sequence
    if current_header is not None:
        sequences[current_header] = ''.join(current_seq_parts)
    elif not sequences:
        raise ValueError('FASTA file is empty')

    return sequences


def write_fasta(
    sequences: dict[str, str],
    filepath: str | Path,
    line_width: int = 60,
) -> None:
    """Write sequences to a FASTA file.

    Parameters
    ----------
    sequences : dict[str, str]
        Mapping of sequence name to sequence string.
    filepath : str or Path
        Output file path. Parent directories must exist.
    line_width : int, optional
        Maximum sequence characters per line. Default is 60.

    Examples
    --------
    >>> import tempfile, pathlib
    >>> with tempfile.NamedTemporaryFile(suffix='.fa', delete=False) as f:
    ...     tmp = f.name
    >>> write_fasta({'seq1': 'ATCG', 'seq2': 'GGGG'}, tmp)
    >>> pathlib.Path(tmp).read_text()
    '>seq1\nATCG\n>seq2\nGGGG\n'
    """
    filepath = Path(filepath)
    with filepath.open('w') as fh:
        for name, seq in sequences.items():
            fh.write(f'>{name}\n')
            for i in range(0, len(seq), line_width):
                fh.write(seq[i:i + line_width] + '\n')


def read_fastq(filepath: str | Path) -> list[tuple[str, str, str]]:
    """Parse a FASTQ file and return reads as a list of tuples.

    Parameters
    ----------
    filepath : str or Path
        Path to the FASTQ file.

    Returns
    -------
    list[tuple[str, str, str]]
        List of (name, sequence, quality) tuples. Name excludes the '@'.

    Raises
    ------
    FileNotFoundError
        If ``filepath`` does not exist.
    ValueError
        If the file is malformed (not a multiple of 4 lines, missing '+' separator).

    Examples
    --------
    >>> import tempfile
    >>> content = '@read1\nATCG\n+\nIIII\n'
    >>> with tempfile.NamedTemporaryFile(mode='w', suffix='.fq', delete=False) as f:
    ...     _ = f.write(content); tmp = f.name
    >>> reads = read_fastq(tmp)
    >>> reads[0]
    ('read1', 'ATCG', 'IIII')
    """
    filepath = Path(filepath)
    if not filepath.exists():
        raise FileNotFoundError(f'FASTQ file not found: {filepath}')

    reads: list[tuple[str, str, str]] = []
    lines = [line.rstrip('\n') for line in filepath.open() if line.strip()]

    if len(lines) % 4 != 0:
        raise ValueError(
            f'FASTQ file has {len(lines)} non-empty lines, expected a multiple of 4'
        )

    for i in range(0, len(lines), 4):
        header = lines[i]
        seq = lines[i + 1]
        plus = lines[i + 2]
        qual = lines[i + 3]
        if not header.startswith('@'):
            raise ValueError(f'Expected @ at line {i + 1}, got: {header!r}')
        if not plus.startswith('+'):
            raise ValueError(f'Expected + at line {i + 3}, got: {plus!r}')
        reads.append((header[1:].split()[0], seq, qual))

    return reads


def parse_bed(filepath: str | Path) -> pd.DataFrame:
    """Parse a BED file into a pandas DataFrame.

    Supports BED3, BED4, BED5, and BED6 formats.

    Parameters
    ----------
    filepath : str or Path
        Path to the BED file. Comment lines (starting with '#' or 'track')
        are skipped.

    Returns
    -------
    pd.DataFrame
        DataFrame with columns: chrom, start (int), end (int), and optional
        name, score, strand depending on the BED format detected.

    Examples
    --------
    >>> import tempfile
    >>> content = 'chr1\t0\t100\tgene1\t0\t+\nchr1\t200\t300\tgene2\t0\t-\n'
    >>> with tempfile.NamedTemporaryFile(mode='w', suffix='.bed', delete=False) as f:
    ...     _ = f.write(content); tmp = f.name
    >>> df = parse_bed(tmp)
    >>> list(df.columns[:3])
    ['chrom', 'start', 'end']
    """
    BED_COLS = ['chrom', 'start', 'end', 'name', 'score', 'strand',
                'thick_start', 'thick_end', 'item_rgb', 'block_count',
                'block_sizes', 'block_starts']

    filepath = Path(filepath)
    rows: list[list[str]] = []
    with filepath.open() as fh:
        for line in fh:
            line = line.rstrip('\n')
            if not line or line.startswith('#') or line.startswith('track') or line.startswith('browser'):
                continue
            rows.append(line.split('\t'))

    if not rows:
        return pd.DataFrame(columns=BED_COLS[:3])

    n_cols = len(rows[0])
    cols = BED_COLS[:n_cols]
    df = pd.DataFrame(rows, columns=cols)
    df['start'] = df['start'].astype(int)
    df['end'] = df['end'].astype(int)
    if 'score' in df.columns:
        df['score'] = pd.to_numeric(df['score'], errors='coerce')
    return df


print('All IO functions defined.')

In [ ]:
import tempfile
import numpy as np

rng = np.random.default_rng(42)

def random_dna(length: int) -> str:
    return ''.join(rng.choice(list('ATCG'), size=length))

# FASTA round-trip test
synthetic_seqs = {f'seq{i:03d}': random_dna(rng.integers(50, 200)) for i in range(20)}

with tempfile.NamedTemporaryFile(mode='w', suffix='.fa', delete=False) as f:
    fasta_path = Path(f.name)

write_fasta(synthetic_seqs, fasta_path)
read_back = read_fasta(fasta_path)

assert set(read_back.keys()) == set(synthetic_seqs.keys()), 'Header mismatch'
for name in synthetic_seqs:
    assert read_back[name] == synthetic_seqs[name], f'Sequence mismatch for {name}'
print(f'FASTA round-trip: PASS ({len(synthetic_seqs)} sequences)')

# FASTQ round-trip test
def write_fastq(reads: list[tuple[str, str, str]], filepath: Path) -> None:
    with filepath.open('w') as fh:
        for name, seq, qual in reads:
            fh.write(f'@{name}\n{seq}\n+\n{qual}\n')

synthetic_reads = [
    (f'read{i:04d}', random_dna(75),
     ''.join(chr(rng.integers(33, 75)) for _ in range(75)))
    for i in range(50)
]

with tempfile.NamedTemporaryFile(mode='w', suffix='.fq', delete=False) as f:
    fastq_path = Path(f.name)

write_fastq(synthetic_reads, fastq_path)
reads_back = read_fastq(fastq_path)

assert len(reads_back) == len(synthetic_reads)
for orig, back in zip(synthetic_reads, reads_back):
    assert orig == back, f'FASTQ round-trip mismatch'
print(f'FASTQ round-trip: PASS ({len(synthetic_reads)} reads)')

# BED parsing test
bed_content = 'chr1\t0\t100\tgene1\t0\t+\nchr1\t200\t300\tgene2\t0\t-\n'
with tempfile.NamedTemporaryFile(mode='w', suffix='.bed', delete=False) as f:
    bed_path = Path(f.name)
    f.write(bed_content)

bed_df = parse_bed(bed_path)
assert len(bed_df) == 2
assert list(bed_df.columns[:3]) == ['chrom', 'start', 'end']
assert bed_df['start'].dtype == int
print(f'BED parse: PASS ({len(bed_df)} intervals)')
print(bed_df)

## Visualization

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
import timeit

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle('compbio_utils.io — File Format Anatomy and Performance', fontsize=13, fontweight='bold')

# --- Panel 1: FASTA format anatomy ---
ax1 = axes[0]
ax1.set_xlim(0, 10)
ax1.set_ylim(0, 10)
ax1.axis('off')
ax1.set_title('FASTA Format Anatomy', fontweight='bold')

fasta_lines = [
    (0.2, 9.0, '>seq1 description text', '#27ae60', 'Header line (> prefix)'),
    (0.2, 7.5, 'ATCGATCGATCGATCGATCG', '#3498db', 'Sequence line 1'),
    (0.2, 6.5, 'GCGCGCGCGCGCGCGCGCGC', '#3498db', 'Sequence line 2 (multi-line)'),
    (0.2, 5.0, '>seq2', '#27ae60', 'Next header'),
    (0.2, 4.0, 'TTTTTTTTTTTTTTTTTTTT', '#3498db', 'Sequence'),
]

for x, y, text, color, annotation in fasta_lines:
    ax1.text(x, y, text, fontsize=9, fontfamily='monospace', color=color, va='center')
    ax1.text(9.8, y, annotation, fontsize=7.5, color='#555', va='center', ha='right')
    ax1.plot([len(text)*0.17 + x, 7.5], [y, y], '--', color='#ccc', lw=0.8)

# --- Panel 2: FASTQ format anatomy ---
ax2 = axes[1]
ax2.set_xlim(0, 10)
ax2.set_ylim(0, 10)
ax2.axis('off')
ax2.set_title('FASTQ Format Anatomy', fontweight='bold')

fastq_lines = [
    (0.2, 9.0, '@read1 instrument info', '#e67e22', 'Header (@ prefix)'),
    (0.2, 7.5, 'ATCGATCGATCGATCGATCG', '#3498db', 'Sequence'),
    (0.2, 6.0, '+', '#95a5a6', 'Separator (+ required)'),
    (0.2, 4.5, 'IIIIIIIIIIIIIIIIIIII', '#e74c3c', 'Quality scores (Phred+33)'),
    (0.2, 3.0, '  = ASCII 73 = Q40 =', '#888', ''),
    (0.2, 2.2, '  0.0001 error prob', '#888', ''),
]

for x, y, text, color, annotation in fastq_lines:
    ax2.text(x, y, text, fontsize=9, fontfamily='monospace', color=color, va='center')
    if annotation:
        ax2.text(9.8, y, annotation, fontsize=7.5, color='#555', va='center', ha='right')

# --- Panel 3: Parsing speed vs file size ---
ax3 = axes[2]
sizes = [10, 50, 200, 500, 1000]
parse_times = []

for n_seqs in sizes:
    seqs = {f's{i}': random_dna(100) for i in range(n_seqs)}
    with tempfile.NamedTemporaryFile(mode='w', suffix='.fa', delete=False) as f:
        tmp_path = Path(f.name)
    write_fasta(seqs, tmp_path)
    t = timeit.timeit(lambda p=tmp_path: read_fasta(p), number=5) / 5 * 1000
    parse_times.append(t)

ax3.plot(sizes, parse_times, 'o-', color='#9b59b6', lw=2, ms=6)
ax3.set_xlabel('Number of sequences in FASTA file')
ax3.set_ylabel('Parse time (ms)')
ax3.set_title('FASTA Parsing Speed', fontweight='bold')
ax3.grid(alpha=0.3)

for x, y in zip(sizes, parse_times):
    ax3.text(x, y + max(parse_times)*0.02, f'{y:.2f} ms', ha='center', fontsize=8)

plt.tight_layout()
plt.savefig('io_module_viz.png', dpi=120, bbox_inches='tight')
plt.show()

## Exercises

See `exercises/README.md` → E10.

1. Write a FASTQ round-trip test: generate 50 synthetic reads, write them, read them back, and assert every field matches.
2. What would happen if `read_fasta` did not handle multi-line sequences? Give an example FASTA file where this would cause a silent error.
3. Extend `parse_bed` to validate that start < end for every interval. Raise a `ValueError` with the line number if not.

## Quiz

1. What is the BED coordinate system (0-based or 1-based)? Is the end position inclusive or exclusive?
2. What does the `+` line in a FASTQ record contain? Can it be non-empty?
3. What is a Phred quality score? What does Q30 mean in terms of error probability?
4. Why does `read_fasta` use `line[1:].split()[0]` to extract the header?
5. What is the difference between `filepath.open()` and `open(filepath)`?

## Reflection

*After completing:* FASTA and FASTQ are simple formats, yet many parsers have subtle bugs. What specific edge cases would you add to a production-quality test suite that the tests here don't cover?

## References

- FASTQ format specification: Cock et al. (2010) Nucleic Acids Research.
- BED format: UCSC Genome Browser documentation.